In [1]:
import os
import logging
from pathlib import Path
from typing import Tuple, Optional, List, Dict
from dataclasses import dataclass
import pandas as pd
import numpy as np
import squidpy as sq
import anndata as ad
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil
import gc
import warnings


@dataclass
class ResourceConfig:
    """Configuration for computational resources."""
    n_jobs: int = -1
    memory_per_job_gb: float = 15.0
    
    def __post_init__(self):
        available_memory_gb = psutil.virtual_memory().available / (1024**3)
        
        if self.n_jobs == -1:
            # Calculate based on memory constraints
            memory_based_jobs = int(available_memory_gb * 0.8 / self.memory_per_job_gb)
            cpu_based_jobs = os.cpu_count()
            self.n_jobs = max(1, min(memory_based_jobs, cpu_based_jobs))
        elif self.n_jobs < 1:
            self.n_jobs = 1


def setup_logging(log_file: Optional[str] = None) -> logging.Logger:
    """Configure logging for the analysis."""
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
    
    return logger


def find_h5ad_files(root_folder: str) -> List[Path]:
    """Recursively find all .h5ad files in the directory tree."""
    root_path = Path(root_folder)
    return list(root_path.rglob("*.h5ad"))


def prepare_spatial_coordinates(
    adata: ad.AnnData,
    coord_keys: Tuple[str, str] = ('x', 'y')
) -> bool:
    """Ensure spatial coordinates are in adata.obsm['spatial']."""
    try:
        if 'spatial' in adata.obsm:
            if adata.obsm['spatial'].shape[1] >= 2:
                return True
        
        if not all(k in adata.obs.columns for k in coord_keys):
            return False
        
        for key in coord_keys:
            if adata.obs[key].isna().any():
                return False
            if not pd.api.types.is_numeric_dtype(adata.obs[key]):
                return False
        
        spatial_coords = adata.obs[list(coord_keys)].values
        adata.obsm['spatial'] = spatial_coords.astype(np.float64)
        
        return True
        
    except Exception:
        return False


def calculate_morans_i_single_file(
    h5ad_path: Path,
    coord_keys: Tuple[str, str],
    output_folder: Path,
    n_neighs: int,
    mode: str,
    n_perms: int
) -> Dict:
    """
    Process a single h5ad file and calculate Moran's I.
    This is the main worker function.
    """
    n_obs_actual = 0
    n_vars_actual = 0
    
    # Force single-threaded execution
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['NUMEXPR_NUM_THREADS'] = '1'
    
    try:
        # Suppress warnings
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            
            # Read data
            adata = ad.read_h5ad(h5ad_path)
            n_obs_actual = adata.shape[0]
            n_vars_actual = adata.shape[1]
            
            # Prepare spatial coordinates
            if not prepare_spatial_coordinates(adata, coord_keys):
                return {
                    "file": h5ad_path.name,
                    "path": str(h5ad_path),
                    "parent_folder": h5ad_path.parent.name,
                    "success": False,
                    "error": "Invalid or missing coordinates",
                    "n_obs": n_obs_actual,
                    "n_vars": n_vars_actual,
                    "n_significant": 0
                }
            
            # Compute spatial neighbors
            sq.gr.spatial_neighbors(
                adata,
                coord_type="generic",
                n_neighs=n_neighs,
                spatial_key="spatial"
            )
            
            # Calculate spatial autocorrelation - CRITICAL: n_jobs=1
            sq.gr.spatial_autocorr(
                adata, 
                mode=mode,
                n_perms=n_perms,
                n_jobs=1
            )
            
            # Extract results
            results_key = "moranI" if mode == "moran" else "gearyC"
            df_results = adata.uns[results_key].copy()
            
            # Rename columns
            column_mapping = {
                'I': f'{mode}_statistic',
                'C': f'{mode}_statistic',
                'pval_norm': 'p_value',
                'pval_z_sim': 'p_value_sim',
                'pval_sim': 'p_value_sim'
            }
            df_results = df_results.rename(columns=column_mapping)
            
            # Add FDR correction
            if 'pval_fdr' not in df_results.columns and 'p_value' in df_results.columns:
                from scipy.stats import false_discovery_control
                df_results['p_value_fdr'] = false_discovery_control(df_results['p_value'])
            
            # Sort by p-value
            if 'p_value' in df_results.columns:
                df_results = df_results.sort_values('p_value')
            
            # Count significant features
            n_significant = 0
            if 'p_value' in df_results.columns:
                n_significant = (df_results["p_value"] < 0.05).sum()
            
            # Save results
            output_csv = output_folder / f"{h5ad_path.stem}_{mode}.csv"
            df_results.to_csv(output_csv, index=True)
            
            # Clean up memory
            del adata, df_results
            gc.collect()
            
            return {
                "file": h5ad_path.name,
                "path": str(h5ad_path),
                "parent_folder": h5ad_path.parent.name,
                "success": True,
                "error": None,
                "n_obs": n_obs_actual,
                "n_vars": n_vars_actual,
                "n_significant": n_significant
            }
        
    except Exception as e:
        gc.collect()
        return {
            "file": h5ad_path.name,
            "path": str(h5ad_path),
            "parent_folder": h5ad_path.parent.name,
            "success": False,
            "error": str(e),
            "n_obs": n_obs_actual,
            "n_vars": n_vars_actual,
            "n_significant": 0
        }


def calculate_spatial_stats_batch(
    root_folder: str,
    coord_keys: Tuple[str, str] = ('x', 'y'),
    output_folder: str = "spatial_results",
    n_neighs: int = 8,
    mode: str = "moran",
    n_perms: int = 1000,
    n_jobs: int = -1,
    memory_per_job_gb: float = 15.0,
    log_file: Optional[str] = None
) -> pd.DataFrame:
    """
    Calculate spatial statistics for all h5ad files.
    
    Parameters
    ----------
    root_folder : str
        Root directory containing .h5ad files
    coord_keys : tuple
        Column names for spatial coordinates in adata.obs
    output_folder : str
        Directory to save results
    n_neighs : int
        Number of spatial neighbors (8 for Cartesian grid)
    mode : str
        Analysis mode: 'moran' for Moran's I or 'geary' for Geary's C
    n_perms : int
        Number of permutations for significance testing
    n_jobs : int
        Number of parallel workers
        -1 = auto-calculate based on memory
        1 = sequential processing
        >1 = specific number of parallel workers
    memory_per_job_gb : float
        Estimated memory per job in GB
    log_file : str, optional
        Path to log file
        
    Returns
    -------
    pd.DataFrame
        Summary of processing results
    """
    # Setup resource configuration
    resource_config = ResourceConfig(
        n_jobs=n_jobs,
        memory_per_job_gb=memory_per_job_gb
    )
    
    # Setup logging
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    logger = setup_logging(log_file)
    
    # System info
    available_memory_gb = psutil.virtual_memory().available / (1024**3)
    total_memory_gb = psutil.virtual_memory().total / (1024**3)
    
    # Log resource configuration
    logger.info(f"System Resources:")
    logger.info(f"  Total CPU cores: {os.cpu_count()}")
    logger.info(f"  Total memory: {total_memory_gb:.2f} GB")
    logger.info(f"  Available memory: {available_memory_gb:.2f} GB")
    logger.info(f"\nJob Configuration:")
    logger.info(f"  Parallel workers: {resource_config.n_jobs}")
    logger.info(f"  Memory per worker: {memory_per_job_gb:.2f} GB")
    logger.info(f"  Total memory needed: {resource_config.n_jobs * memory_per_job_gb:.2f} GB")
    logger.info(f"  Permutations: {n_perms}")
    
    # Find files
    h5ad_files = find_h5ad_files(root_folder)
    logger.info(f"\nFound {len(h5ad_files)} AnnData files in {root_folder}")
    
    if not h5ad_files:
        logger.warning("No .h5ad files found!")
        return pd.DataFrame()
    
    # Process files
    logger.info(f"\nStarting processing...")
    logger.info("="*70)
    
    results = []
    
    if resource_config.n_jobs == 1:
        # Sequential processing
        logger.info("Running in SEQUENTIAL mode (n_jobs=1)\n")
        for i, h5ad_path in enumerate(h5ad_files, 1):
            logger.info(f"[{i}/{len(h5ad_files)}] Processing {h5ad_path.name}...")
            result = calculate_morans_i_single_file(
                h5ad_path, coord_keys, output_path, n_neighs, mode, n_perms
            )
            results.append(result)
            
            if result['success']:
                logger.info(f"  ✓ Success: {result['n_significant']} significant features")
            else:
                logger.info(f"  ✗ Failed: {result['error']}")
    else:
        # Parallel processing using ProcessPoolExecutor
        logger.info(f"Running in PARALLEL mode with {resource_config.n_jobs} workers\n")
        
        # Submit all tasks
        with ProcessPoolExecutor(max_workers=resource_config.n_jobs) as executor:
            # Submit all jobs
            future_to_file = {
                executor.submit(
                    calculate_morans_i_single_file,
                    h5ad_path, coord_keys, output_path, n_neighs, mode, n_perms
                ): h5ad_path
                for h5ad_path in h5ad_files
            }
            
            # Process completed tasks
            completed = 0
            for future in as_completed(future_to_file):
                completed += 1
                result = future.result()
                results.append(result)
                
                status = "✓" if result['success'] else "✗"
                if result['success']:
                    logger.info(
                        f"[{completed}/{len(h5ad_files)}] {status} {result['file']} "
                        f"({result['n_significant']} significant, "
                        f"{result['n_obs']} obs, {result['n_vars']} vars)"
                    )
                else:
                    logger.info(f"[{completed}/{len(h5ad_files)}] {status} {result['file']} - {result['error']}")
    
    # Create summary DataFrame
    summary_df = pd.DataFrame(results)
    success_count = summary_df["success"].sum()
    
    # Log final summary
    logger.info(f"\n{'='*70}")
    logger.info(f"PROCESSING COMPLETE")
    logger.info(f"{'='*70}")
    logger.info(f"Success rate: {success_count}/{len(h5ad_files)} files ({success_count/len(h5ad_files)*100:.1f}%)")
    
    if 'n_significant' in summary_df.columns and success_count > 0:
        total_sig = summary_df[summary_df['success']]['n_significant'].sum()
        avg_sig = summary_df[summary_df['success']]['n_significant'].mean()
        logger.info(f"Total significant features: {total_sig}")
        logger.info(f"Average per file: {avg_sig:.1f}")
    
    # Summary by folder
    if 'parent_folder' in summary_df.columns:
        logger.info("\nResults by folder:")
        folder_summary = summary_df.groupby('parent_folder').agg({
            'success': ['sum', 'count'],
            'n_significant': 'sum'
        })
        logger.info(f"\n{folder_summary}")
    
    # Save summary
    summary_path = output_path / "processing_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    logger.info(f"\nDetailed summary saved to: {summary_path}")
    
    # Log failed files
    failed = summary_df[~summary_df['success']]
    if len(failed) > 0:
        logger.info(f"\n{len(failed)} files failed:")
        for _, row in failed.iterrows():
            logger.info(f"  - {row['file']}: {row['error']}")
    
    return summary_df


def estimate_processing_time(
    sample_file: str,
    n_files: int,
    n_jobs: int = -1,
    n_perms: int = 1000,
    memory_per_job_gb: float = 15.0
) -> Dict[str, float]:
    """Estimate processing time by running on a sample file."""
    import time
    
    available_memory_gb = psutil.virtual_memory().available / (1024**3)
    
    if n_jobs == -1:
        memory_based_jobs = int(available_memory_gb * 0.8 / memory_per_job_gb)
        cpu_based_jobs = os.cpu_count()
        n_jobs = max(1, min(memory_based_jobs, cpu_based_jobs))
    
    print(f"Estimating processing time using {sample_file}...")
    print(f"Configuration: {n_jobs} workers, {n_perms} permutations")
    print(f"Available memory: {available_memory_gb:.1f}GB\n")
    
    start = time.time()
    
    # Create temporary output directory
    temp_output = Path("temp_estimate_output")
    temp_output.mkdir(exist_ok=True)
    
    result = calculate_morans_i_single_file(
        Path(sample_file),
        ('x', 'y'),
        temp_output,
        8,
        "moran",
        n_perms
    )
    
    elapsed = time.time() - start
    
    if not result['success']:
        print(f"Error: {result['error']}")
        return {"error": result['error']}
    
    sequential_time = elapsed * n_files / 60
    parallel_time = sequential_time / n_jobs
    
    print(f"\n{'='*50}")
    print(f"Estimation Results:")
    print(f"{'='*50}")
    print(f"File shape: ({result['n_obs']}, {result['n_vars']})")
    print(f"Single file time: {elapsed:.1f} seconds")
    print(f"Sequential time: {sequential_time:.1f} minutes")
    print(f"Parallel time: {parallel_time:.1f} minutes")
    print(f"Speedup: {n_jobs}x")
    print(f"{'='*50}")
    
    # Clean up temp files
    import shutil
    shutil.rmtree(temp_output, ignore_errors=True)
    
    return {
        "single_file_seconds": elapsed,
        "estimated_sequential_minutes": sequential_time,
        "estimated_parallel_minutes": parallel_time,
        "n_jobs_used": n_jobs,
        "speedup_factor": n_jobs
    }


if __name__ == "__main__":
    # Run directly (not recommended in Jupyter - save as .py file)
    summary = calculate_spatial_stats_batch(
        root_folder="/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/",
        coord_keys=('x', 'y'),
        output_folder="/home/ajarrah/PhD_Thesis/chapter_4/results/spatial_results",
        n_neighs=8,
        mode="moran",
        n_perms=1000,
        n_jobs=-1,
        memory_per_job_gb=15.0,
        log_file="/home/ajarrah/PhD_Thesis/chapter_4/results/spatial_analysis.log"
    )
    
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    print(f"Success rate: {summary['success'].sum()}/{len(summary)} files")
    if summary['success'].sum() > 0:
        print(f"Total significant features: {summary[summary['success']]['n_significant'].sum()}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
2025-10-29 15:46:08,886 - INFO - System Resources:
2025-10-29 15:46:08,887 - INFO -   Total CPU cores: 80
2025-10-29 15:46:08,887 - INFO -   Total memory: 187.53 GB
2025-10-29 15:46:08,888 - INFO -   Available memory: 179.49 GB
2025-10-29 15:46:08,889 - INFO - 
Job Configuration:
2025-10-29 15:46:08,889 - INFO -   Parallel workers: 9
2025-10-29 15:46:08,890 - INFO -   Memory per worker: 15.00 GB
2025-10-29 15:46:08,891 - INFO -   Total memory needed: 135.00 GB
2025-10-29 15:46:08,891 - INFO -   Permutations: 1000
2025-10-29 15:46:08,896 - INFO - 
Found 48 AnnData files in /home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/
2025-10-29 15:46:08,896 - INFO - 
Starting processing...
2025-10-29 15:46:08,896 - INFO - ======================================================================
2025-10-29


FINAL SUMMARY
Success rate: 48/48 files
Total significant features: 143744
